# HoneyAI: KYPO real-human check, timing calibration, baselines (Colab)

No GPU needed (CPU runtime is fine, about 5 minutes in total).

1. Converts the **KYPO** dataset (real humans, CC BY 4.0) into HoneyAI sessions.
2. **Calibrates** the simulated humans' think time to KYPO (quick commands on target hosts).
3. Re-generates the calibrated dataset **`data/v2b`** (must match the shipped copy byte for byte).
4. Runs the **real-human check** and **all non-LLM baselines** with the same metrics as the LLM notebook.

When the first cell asks, upload **both** `honeyai_code.zip` and `honeyai_datasets.zip` (select both files in the dialog).

In [ ]:
from google.colab import files
up = files.upload()                      # choose honeyai_code.zip AND honeyai_datasets.zip
!unzip -q -o honeyai_code.zip -d honeyai
!unzip -q -o honeyai_datasets.zip -d honeyai
%cd honeyai
!pip install -q xgboost
!ls data external/kypo

In [ ]:
# KYPO raw data: Svabensky et al., Data in Brief 38 (2021), doi:10.1016/j.dib.2021.107398 (CC BY 4.0)
!curl -sSL -o external/kypo/data.zip "https://zenodo.org/records/8136017/files/data.zip?download=1"
!ls -la external/kypo

In [ ]:
# 1) KYPO -> sessions (commands + timing only; KYPO has no outputs)
!python scripts/convert_kypo.py

In [ ]:
# 2) fit simulated-human think time to KYPO (about 1 minute)
!python scripts/calibrate_human_timing.py

In [ ]:
# 3) re-generate v2b into a scratch folder and check it matches the shipped copy
!python scripts/generate_sessions.py --human-timing configs/human_timing.json --out data/v2b_check
!cmp data/v2b_check/dataset1_sessions.json data/v2b/dataset1_sessions.json && echo 'v2b reproduced exactly'

In [ ]:
# 4) real-human check + all non-LLM baselines on v2b
!python scripts/eval_kypo.py --train data/v2b/dataset1_sessions.json --out runs/kypo_eval_v2b.json
!python scripts/run_baselines.py --data data/v2b/dataset1_sessions.json --out runs/baselines_v2b.json

In [ ]:
# 5) download the results
!zip -qr honeyai_colab_outputs.zip runs configs/human_timing.json
files.download("honeyai_colab_outputs.zip")

## Next: the LLM on the same data (GPU runtime)

Open `notebooks/qlora_generative_patched.ipynb` and set:
- main run: `DATA_PATH = "data/v2b/dataset4_llm_sequences.jsonl"`
- ablation + real-human check: also `COMMANDS_ONLY = True` and `EXTERNAL_TEST_PATH = "external/kypo/kypo_sessions_ds4_commands_only.jsonl"`

Report the KYPO numbers as an **external check on real humans with a different task set**, not as test accuracy.